In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
qc_df.to_csv(OUTPUT_DIR / f"database_qc_results_{timestamp}.csv", index=False)
qc_df.to_json(OUTPUT_DIR / f"database_qc_results_{timestamp}.json", orient='records', indent=2)
print(f"Saved QC outputs to {OUTPUT_DIR}")

Saved QC outputs to /Users/matthewpitlock/Development/repos/magicmatt_fantasysports/fantasy_baseball_roster_management/notebooks/artifacts


: 

In [11]:
qc_results = []
qc_results.append({
    'check_name': 'dim_players_missing_primary_position',
    'table_name': 'dim_players',
    'metric_value': int(conn.execute("SELECT COUNT(*) FROM dim_players WHERE primary_position IS NULL").fetchone()[0]),
    'threshold': 0,
    'severity': 'high',
    'status': 'warn',
})
qc_results.append({
    'check_name': 'dim_players_missing_primary_team',
    'table_name': 'dim_players',
    'metric_value': int(conn.execute("SELECT COUNT(*) FROM dim_players WHERE primary_team IS NULL").fetchone()[0]),
    'threshold': 0,
    'severity': 'high',
    'status': 'warn',
})
qc_results.append({
    'check_name': 'dim_players_total_rows',
    'table_name': 'dim_players',
    'metric_value': int(conn.execute("SELECT COUNT(*) FROM dim_players").fetchone()[0]),
    'threshold': 0,
    'severity': 'info',
    'status': 'pass',
})

qc_df = pd.DataFrame(qc_results)
qc_df

,check_name,table_name,metric_value,threshold,severity,status
0,dim_players_missing_primary_position,dim_players,7168,0,high,warn
1,dim_players_missing_primary_team,dim_players,7168,0,high,warn
2,dim_players_total_rows,dim_players,7168,0,info,pass


In [10]:
dim_players_nulls = conn.execute("""
SELECT player_id, player_name, primary_position, primary_team
FROM dim_players
WHERE primary_position IS NULL OR primary_team IS NULL
ORDER BY player_id
LIMIT 50
""").fetchdf()
dim_players_nulls

,player_id,player_name,primary_position,primary_team
0,1,2027 Early 1st,None,None
1,2,2027 Early 2nd,None,None
2,3,2027 Early 3rd,None,None
3,4,2027 Late 1st,None,None
4,5,2027 Late 2nd,None,None
5,6,2027 Late 3rd,None,None
6,7,2027 Mid 1st,None,None
7,8,2027 Mid 2nd,None,None
8,9,2027 Mid 3rd,None,None
9,10,2028 Early 1st,None,None


In [9]:
team_rows = []
for table in ['stg_fantrax_status', 'stg_harry_rankings', 'stg_keith_prospects', 'fact_player_status', 'fact_player_stats']:
    try:
        cols = [c[1] for c in conn.execute(f"PRAGMA table_info('{table}')").fetchall()]
        if 'player_name' in cols and 'team' in cols:
            team_counts = conn.execute(f"""
            SELECT player_name, COUNT(DISTINCT team) AS distinct_team_count
            FROM {table}
            WHERE player_name IS NOT NULL AND team IS NOT NULL
            GROUP BY player_name
            ORDER BY distinct_team_count DESC, player_name
            LIMIT 20
            """).fetchdf()
            if not team_counts.empty:
                team_rows.append({'table_name': table, 'sample': team_counts})
    except Exception:
        continue

team_rows[0]['sample'] if team_rows else None

,player_name,distinct_team_count
0,Carlos Martinez,4
1,Gabriel Rodriguez,4
2,Carlos Rodriguez,3
3,Carlos Sanchez,3
4,Jose Ramos,3
5,Jose Rodriguez,3
6,Luis De Leon,3
7,Luis Garcia,3
8,Alexis Hernandez,2
9,Angel Guzman,2


In [8]:
player_checks = []

# Duplicate player_id values
for table in ['dim_players', 'player_aliases', 'fact_contracts', 'fact_player_status', 'fact_player_stats', 'fact_harry_rankings', 'fact_keith_rankings']:
    try:
        cols = [c[1] for c in conn.execute(f"PRAGMA table_info('{table}')").fetchall()]
        if 'player_id' in cols:
            dup_df = conn.execute(f"SELECT player_id, COUNT(*) AS row_count FROM {table} GROUP BY player_id HAVING COUNT(*) > 1").fetchdf()
            if not dup_df.empty:
                player_checks.append({
                    'table_name': table,
                    'check_name': 'duplicate_player_id',
                    'issue_count': len(dup_df),
                    'affected_rows': int(dup_df['row_count'].sum()),
                })
    except Exception:
        continue

# Duplicate player names in dim_players
try:
    dim_players_dup = conn.execute("""
    SELECT player_name, COUNT(*) AS row_count
    FROM dim_players
    GROUP BY player_name
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC, player_name
    """).fetchdf()
    if not dim_players_dup.empty:
        player_checks.append({
            'table_name': 'dim_players',
            'check_name': 'duplicate_player_name',
            'issue_count': len(dim_players_dup),
            'affected_rows': int(dim_players_dup['row_count'].sum()),
        })
except Exception as exc:
    print(f"dim_players duplicate-name check skipped: {exc}")

pd.DataFrame(player_checks).sort_values(['affected_rows', 'issue_count'], ascending=False)

,table_name,check_name,issue_count,affected_rows
0,player_aliases,duplicate_player_id,7021,16426
3,fact_player_stats,duplicate_player_id,89,189
2,fact_player_status,duplicate_player_id,88,187
1,fact_contracts,duplicate_player_id,11,22
4,fact_harry_rankings,duplicate_player_id,3,6


In [7]:
duplicate_rows = []
for table in tables:
    cols = [c[1] for c in conn.execute(f"PRAGMA table_info('{table}')").fetchall()]
    if not cols:
        continue
    cols_sql = ', '.join(cols)
    q = f"""
    SELECT {cols_sql}, COUNT(*) AS duplicate_count
    FROM {table}
    GROUP BY {cols_sql}
    HAVING COUNT(*) > 1
    """
    dup_df = conn.execute(q).fetchdf()
    if not dup_df.empty:
        duplicate_rows.append({
            'table_name': table,
            'duplicate_groups': len(dup_df),
            'affected_rows': int(dup_df['duplicate_count'].sum()),
        })

if duplicate_rows:
    pd.DataFrame(duplicate_rows).sort_values('affected_rows', ascending=False)
else:
    pd.DataFrame(columns=['table_name', 'duplicate_groups', 'affected_rows'])

In [5]:
null_rows = []
for table in tables:
    columns = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    for col in columns:
        col_name = col[1]
        null_count = conn.execute(f"SELECT COUNT(*) FROM {table} WHERE {col_name} IS NULL").fetchone()[0]
        null_pct = round(100 * null_count / max(1, conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]), 2)
        null_rows.append({
            'table_name': table,
            'column_name': col_name,
            'null_count': null_count,
            'null_pct': null_pct,
        })

null_df = pd.DataFrame(null_rows)
null_df = null_df[null_df['null_count'] > 0].sort_values(['null_count', 'table_name'], ascending=[False, True])
null_df.head(200)

,table_name,column_name,null_count,null_pct
3,dim_players,primary_position,7168,100.00
4,dim_players,primary_team,7168,100.00
58,player_aliases,source_player_id,2336,14.10
105,stg_harry_rankings,level,26,1.49
102,stg_harry_rankings,age,18,1.03
103,stg_harry_rankings,positions,18,1.03
104,stg_harry_rankings,team,18,1.03
16,fact_contracts,free_agent_year,13,2.57
68,stg_contract_tracker,free_agent_year,13,2.62
92,stg_fantrax_status,opponent,3,0.04


In [4]:
summary_rows = []
for table in tables:
    row_count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    columns = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    pk_cols = [c[1] for c in columns if c[5] == 1]
    summary_rows.append({
        'table_name': table,
        'row_count': row_count,
        'column_count': len(columns),
        'pk_columns': ', '.join(pk_cols) if pk_cols else 'None',
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.sort_values('row_count', ascending=False)

,table_name,row_count,column_count,pk_columns
9,player_aliases,16573,7,alias_id
0,dim_players,7168,7,player_id
6,fact_player_stats,7119,9,player_stats_id
7,fact_player_status,7118,6,player_status_id
11,stg_fantrax_stats,7104,11,None
12,stg_fantrax_status,7103,14,None
4,fact_harry_rankings,1760,6,harry_rank_id
13,stg_harry_rankings,1746,10,None
3,fact_contracts,505,8,contract_id
8,fact_roster_snapshots,505,7,roster_snapshot_id


In [3]:
create_statements = {}
for table in tables:
    try:
        ddl_row = conn.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name=?", (table,)).fetchone()
        if ddl_row:
            create_statements[table] = ddl_row[0]
    except Exception as exc:
        create_statements[table] = f"ERROR: {exc}"

create_statements_df = pd.DataFrame({
    'table_name': list(create_statements.keys()),
    'create_statement': list(create_statements.values()),
})
create_statements_df

,table_name,create_statement
0,dim_players,CREATE TABLE dim_players(player_id INTEGER PRI...
1,dim_positions,CREATE TABLE dim_positions(position_id INTEGER...
2,dim_teams,CREATE TABLE dim_teams(team_id INTEGER PRIMARY...
3,fact_contracts,CREATE TABLE fact_contracts(contract_id INTEGE...
4,fact_harry_rankings,CREATE TABLE fact_harry_rankings(harry_rank_id...
5,fact_keith_rankings,CREATE TABLE fact_keith_rankings(keith_rank_id...
6,fact_player_stats,CREATE TABLE fact_player_stats(player_stats_id...
7,fact_player_status,CREATE TABLE fact_player_status(player_status_...
8,fact_roster_snapshots,CREATE TABLE fact_roster_snapshots(roster_snap...
9,player_aliases,CREATE TABLE player_aliases(alias_id INTEGER P...


In [2]:
tables_df = sql_df("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
  AND table_type = 'BASE TABLE'
ORDER BY table_name
""")

tables = tables_df['table_name'].tolist()
tables_df

,table_name
0,dim_players
1,dim_positions
2,dim_teams
3,fact_contracts
4,fact_harry_rankings
5,fact_keith_rankings
6,fact_player_stats
7,fact_player_status
8,fact_roster_snapshots
9,player_aliases


In [1]:
import os
import re
import json
import pandas as pd
import duckdb
from pathlib import Path
from datetime import datetime

DB_PATH = "/Users/matthewpitlock/Development/FantasyBaseballAnalytics/database/sda_fantasybaseball.db"
OUTPUT_DIR = Path("/Users/matthewpitlock/Development/repos/magicmatt_fantasysports/fantasy_baseball_roster_management/notebooks/artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conn = duckdb.connect(DB_PATH)


def sql_df(query: str, params: tuple | None = None) -> pd.DataFrame:
    if params is None:
        return conn.execute(query).fetchdf()
    return conn.execute(query, params).fetchdf()


print(f"Connected to {DB_PATH}")
print(f"Notebook artifacts directory: {OUTPUT_DIR}")

Connected to /Users/matthewpitlock/Development/FantasyBaseballAnalytics/database/sda_fantasybaseball.db
Notebook artifacts directory: /Users/matthewpitlock/Development/repos/magicmatt_fantasysports/fantasy_baseball_roster_management/notebooks/artifacts


# Database QC Review

This notebook performs an initial quality-control review of the DuckDB warehouse tables, including schema inspection, null profiling, duplicate detection, player-integrity checks, and team-cardinality review.